In [3]:
"""
delta_optimizer.py
==================
Ottimizzazione tabelle Delta Lake in base al livello Medallion Architecture
e allo scopo di visualizzazione del dato.

Compatibile con: Microsoft Fabric (Spark), Databricks

Autore: Marco Pozzan
"""

from pyspark.sql import SparkSession
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
from datetime import datetime, timedelta
import logging
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("DeltaOptimizer")


# ─────────────────────────────────────────────
#  ENUM: Livelli Medallion
# ─────────────────────────────────────────────
class MedallionLayer(str, Enum):
    BRONZE = "bronze"   # Raw / Landing — write-heavy, read-rarely
    SILVER = "silver"   # Cleansed / Conformed — bilanciato
    GOLD   = "gold"     # Aggregated / Serving — read-heavy, multi-consumer


# ─────────────────────────────────────────────
#  ENUM: Scopo di visualizzazione
# ─────────────────────────────────────────────
class VisualizationPurpose(str, Enum):
    OPERATIONAL    = "operational"    # Report operativi real-time / near real-time
    ANALYTICAL     = "analytical"     # Analisi storiche, trend, drill-down
    DASHBOARD_KPI  = "dashboard_kpi"  # KPI su Power BI / Direct Lake
    EXPORT         = "export"         # Export bulk verso altri sistemi
    ML_TRAINING    = "ml_training"    # Feature store per modelli ML
    ARCHIVE        = "archive"        # Storico freddo, accesso raro


# ─────────────────────────────────────────────
#  ENUM: Strategia multi-consumer (Gold)
# ─────────────────────────────────────────────
class ConsumerStrategy(str, Enum):
    SINGLE_TABLE    = "single_table"    # Compromesso ragionevole: 400MB + V-Order
    DEDICATED_TABLES = "dedicated"      # Duplicazione: tabella Spark + tabella Power BI
    SEMANTIC_MODEL  = "semantic_model"  # Gold per Spark, Power BI via Semantic Model


# ─────────────────────────────────────────────
#  PROFILO DI OTTIMIZZAZIONE
# ─────────────────────────────────────────────
@dataclass
class OptimizationProfile:

    # ── OPTIMIZE / Z-ORDER ──────────────────────────────────────────────────
    run_optimize:            bool = True
    zorder_columns:          list[str] = field(default_factory=list)
    vorder_optimize:         bool = False    # OPTIMIZE ... VORDER (Fabric)
    optimize_dry_run_first:  bool = False    # esegue DRY RUN prima, salta se già compatto
    optimize_hot_partitions_only: bool = False  # OPTIMIZE WHERE date >= N giorni fa
    hot_partition_days:      int  = 14       # finestra "hot" in giorni

    # ── VACUUM ──────────────────────────────────────────────────────────────
    run_vacuum:              bool = True
    vacuum_retain_hours:     int  = 168      # default 7 giorni

    # ── AUTO OPTIMIZE (optimizeWrite) ───────────────────────────────────────
    auto_optimize:           bool = False    # delta.autoOptimize.optimizeWrite

    # ── AUTO COMPACT (legacy flag) ───────────────────────────────────────────
    auto_compact:            bool = False    # delta.autoOptimize.autoCompact

    # ── AUTO COMPACT avanzato (Microsoft Fabric) ─────────────────────────────
    auto_compact_enabled:    bool = False    # delta.autoCompact.enabled
    auto_compact_min_files:  int  = 10       # delta.autoCompact.minNumFiles

    # ── TARGET FILE SIZE ────────────────────────────────────────────────────
    target_file_size_mb:     Optional[int] = None

    # ── PARTITION ───────────────────────────────────────────────────────────
    partition_hint:          Optional[str] = None
    partition_column:        Optional[str] = None  # colonna data per hot-partition filter

    # ── DATA SKIPPING / STATS ───────────────────────────────────────────────
    data_skipping_columns:   list[str] = field(default_factory=list)
    stats_columns:           int  = 32       # delta.dataSkippingNumIndexedCols

    # ── DELETION VECTORS ────────────────────────────────────────────────────
    enable_deletion_vectors: bool = False    # delta.enableDeletionVectors

    # ── V-ORDER (Microsoft Fabric) ──────────────────────────────────────────
    vorder_enabled:          bool = False    # delta.parquet.vorder.enabled

    # ── LOG RETENTION ───────────────────────────────────────────────────────
    log_retention_days:      Optional[int] = None   # delta.logRetentionDuration

    # ── CHANGE DATA FEED ────────────────────────────────────────────────────
    enable_cdf:              bool = False    # delta.enableChangeDataFeed

    # ── SCHEMA EVOLUTION (Bronze) ───────────────────────────────────────────
    enable_column_mapping:   bool = False    # delta.columnMapping.mode = name

    # ── CHECKPOINT INTERVAL (Silver streaming) ──────────────────────────────
    checkpoint_interval:     Optional[int] = None   # delta.checkpointInterval

    # ── LIQUID CLUSTERING ───────────────────────────────────────────────────
    enable_liquid_clustering: bool = False
    liquid_cluster_columns:   list[str] = field(default_factory=list)
    table_size_gb:            Optional[float] = None  # usato per warning su tabelle piccole

    # ── WAREHOUSE MODE ───────────────────────────────────────────────────────
    warehouse_mode:          bool = False    # se True: skip OPTIMIZE/VACUUM (gestito da Fabric)

    # ── NOTE descrittive ────────────────────────────────────────────────────
    notes: str = ""


# ─────────────────────────────────────────────
#  MATRICE DI STRATEGIA
# ─────────────────────────────────────────────
def get_optimization_profile(
    layer: MedallionLayer,
    purpose: VisualizationPurpose,
    zorder_columns: Optional[list[str]] = None,
    partition_column: Optional[str] = None,
    consumer_strategy: ConsumerStrategy = ConsumerStrategy.SINGLE_TABLE,
    warehouse_mode: bool = False,
    table_size_gb: Optional[float] = None,
) -> OptimizationProfile:
    """
    Restituisce un OptimizationProfile configurato in base a layer + purpose.

    Parameters
    ----------
    layer             : livello Medallion (BRONZE / SILVER / GOLD)
    purpose           : scopo di utilizzo del dato
    zorder_columns    : colonne per Z-ORDER (es. ["DataOrdine", "ClienteID"])
    partition_column  : colonna data di partizione (es. "event_date")
    consumer_strategy : per Gold, strategia multi-consumer
    warehouse_mode    : se True, tabella gestita da Fabric Warehouse (no OPTIMIZE/VACUUM)
    table_size_gb     : dimensione tabella in GB (per decisioni Liquid Clustering)
    """

    zo = zorder_columns or []
    p = OptimizationProfile()
    p.partition_column = partition_column
    p.warehouse_mode   = warehouse_mode
    p.table_size_gb    = table_size_gb

    # ── WAREHOUSE MODE: manutenzione delegata a Fabric ──────────────────────
    if warehouse_mode:
        p.run_optimize = False
        p.run_vacuum   = False
        p.notes = (
            "Warehouse mode: OPTIMIZE e VACUUM non supportati. "
            "Fabric gestisce auto-compaction e checkpoint automaticamente."
        )
        return p

    # ════════════════════════════════════════════════════════════════════════
    #  BRONZE
    # ════════════════════════════════════════════════════════════════════════
    if layer == MedallionLayer.BRONZE:

        if purpose == VisualizationPurpose.ARCHIVE:
            # Storico raw — dati letti 1 sola volta da Silver, poi mai
            p.run_optimize          = False   # spreco su dati write-only
            p.vacuum_retain_hours   = 48      # retention breve (24-48h) — dati raw
            p.auto_compact_enabled  = False   # write-heavy: compaction = overhead inutile
            p.auto_optimize         = False
            p.target_file_size_mb   = 256
            p.log_retention_days    = 7       # retention breve su Bronze
            p.vorder_enabled        = False   # overhead non giustificato
            p.enable_cdf            = True    # CDF per Silver CDC
            p.enable_column_mapping = True    # schema evolution flessibile
            p.notes = (
                "Bronze/Archive: NO auto-compaction (write-heavy, dati letti 1x). "
                "CDF abilitato per propagazione Silver. VACUUM aggressivo 48h."
            )

        else:
            # Bronze operativo (streaming, batch ingestion)
            p.run_optimize          = False   # NO OPTIMIZE schedulato su Bronze
            p.vacuum_retain_hours   = 48      # aggressivo — dati raw poco preziosi
            p.auto_optimize         = True    # optimizeWrite: file decenti già in ingestion
            p.auto_compact_enabled  = False   # Bronze è write-heavy, compaction = spreco
            p.target_file_size_mb   = 128
            p.log_retention_days    = 7
            p.vorder_enabled        = False   # Bronze non esposto a Power BI
            p.enable_cdf            = True    # CDF per Silver CDC
            p.enable_column_mapping = True
            p.notes = (
                "Bronze/Operational: optimizeWrite attivo, NO auto-compaction. "
                "VACUUM aggressivo 48h. CDF per Silver."
            )

    # ════════════════════════════════════════════════════════════════════════
    #  SILVER
    # ════════════════════════════════════════════════════════════════════════
    elif layer == MedallionLayer.SILVER:

        if purpose == VisualizationPurpose.OPERATIONAL:
            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vacuum_retain_hours        = 168      # 7 giorni
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 5        # compaction aggressiva per bassa latenza
            p.target_file_size_mb        = 64       # file piccoli → latenza bassa
            p.enable_deletion_vectors    = True
            p.log_retention_days         = 14
            p.vorder_enabled             = False    # Silver ancora in trasformazione
            p.enable_cdf                 = True     # CDF per Gold CDC
            p.checkpoint_interval        = 100
            p.optimize_hot_partitions_only = True
            p.hot_partition_days         = 14
            p.notes = (
                "Silver/Operational: file 64MB, DV per UPSERT veloci, "
                "OPTIMIZE sulle ultime 2 settimane, CDF per Gold."
            )

        elif purpose == VisualizationPurpose.ANALYTICAL:
            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vacuum_retain_hours        = 168      # 7 giorni
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 10
            p.target_file_size_mb        = 256
            p.stats_columns              = 32
            p.log_retention_days         = 30
            p.vorder_enabled             = False    # Silver: no V-ORDER (ancora trasformazioni)
            p.enable_cdf                 = True
            p.checkpoint_interval        = 100
            p.optimize_dry_run_first     = True     # controlla se auto-compact ha già compattato
            p.optimize_hot_partitions_only = True
            p.hot_partition_days         = 14
            p.notes = (
                "Silver/Analytical: file 256MB, OPTIMIZE su hot partitions (14gg), "
                "DRY RUN prima per evitare doppia compaction."
            )

        elif purpose == VisualizationPurpose.ML_TRAINING:
            p.run_optimize               = True
            p.zorder_columns             = []       # ML legge tutto, Z-ORDER non utile
            p.vacuum_retain_hours        = 168
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 20       # soglia alta → meno interruzioni ingestion
            p.target_file_size_mb        = 512      # file grandi → throughput scan massimo
            p.stats_columns              = 8        # meno stats overhead
            p.log_retention_days         = 14
            p.vorder_enabled             = False    # scan sequenziale: V-ORDER = overhead inutile
            p.enable_cdf                 = False    # feature store: no CDF
            p.notes = (
                "Silver/ML: file 512MB per throughput massimo, "
                "NO V-ORDER (scan sequenziale), NO Z-ORDER."
            )

        else:
            # Silver generico
            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vacuum_retain_hours        = 168
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 10
            p.log_retention_days         = 14
            p.vorder_enabled             = False
            p.enable_cdf                 = True
            p.notes = "Silver: profilo generico bilanciato."

    # ════════════════════════════════════════════════════════════════════════
    #  GOLD
    # ════════════════════════════════════════════════════════════════════════
    elif layer == MedallionLayer.GOLD:

        if purpose == VisualizationPurpose.DASHBOARD_KPI:
            # Compromesso ragionevole (SINGLE_TABLE) o ottimizzazione dedicata
            if consumer_strategy == ConsumerStrategy.DEDICATED_TABLES:
                # Profilo per tabella dedicata Power BI (600MB, V-Order, row groups 8M+)
                p.target_file_size_mb = 600
                p.notes_suffix = "Tabella dedicata Power BI (600MB, VORDER). Sync da gold_master."
            else:
                # Compromesso: 400MB + V-Order — SQL felice, Power BI accettabile
                p.target_file_size_mb = 400

            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vorder_optimize            = True     # OPTIMIZE ... VORDER
            p.vacuum_retain_hours        = 720      # 30 giorni — compliance
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 10
            p.enable_deletion_vectors    = True
            p.stats_columns              = 32
            p.log_retention_days         = 90       # compliance / audit
            p.vorder_enabled             = True     # V-ORDER critico per Direct Lake
            p.enable_cdf                 = False    # Gold = layer finale
            p.optimize_dry_run_first     = True

            # Liquid Clustering: solo se tabella > 100GB
            if zo and table_size_gb and table_size_gb >= 100:
                p.enable_liquid_clustering = True
                p.liquid_cluster_columns   = zo[:2]
            elif zo and (not table_size_gb or table_size_gb < 100):
                log.warning(
                    f"    Liquid Clustering richiesto ma tabella <100GB "
                    f"({table_size_gb or 'sconosciuta'} GB): usa Z-ORDER, costo inferiore."
                )

            p.notes = (
                f"Gold/Dashboard KPI ({consumer_strategy.value}): "
                f"V-ORDER + DV per Direct Lake, retention 30gg, log 90gg. "
                f"{'Liquid Clustering abilitato (>100GB).' if p.enable_liquid_clustering else 'Z-ORDER (tabella <100GB).'}"
            )

        elif purpose == VisualizationPurpose.ANALYTICAL:
            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vorder_optimize            = True
            p.vacuum_retain_hours        = 720      # 30 giorni
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 10
            p.target_file_size_mb        = 400      # compromesso SQL + Power BI
            p.stats_columns              = 32
            p.log_retention_days         = 90
            p.vorder_enabled             = True
            p.enable_cdf                 = False
            p.optimize_dry_run_first     = True
            p.notes = "Gold/Analytical: 400MB, V-ORDER, Z-ORDER su chiavi analitiche, retention 30gg."

        elif purpose == VisualizationPurpose.EXPORT:
            p.run_optimize               = True
            p.zorder_columns             = []
            p.vorder_optimize            = False    # export: formato grezzo
            p.vacuum_retain_hours        = 72       # breve, tabella ricreata spesso
            p.auto_optimize              = False
            p.auto_compact_enabled       = False
            p.target_file_size_mb        = 512      # file grandi → throughput export
            p.stats_columns              = 4
            p.log_retention_days         = 7
            p.vorder_enabled             = False
            p.enable_cdf                 = False
            p.notes = "Gold/Export: file 512MB, NO V-ORDER, retention breve."

        elif purpose == VisualizationPurpose.OPERATIONAL:
            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vorder_optimize            = True
            p.vacuum_retain_hours        = 72
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 5        # compaction aggressiva
            p.target_file_size_mb        = 64       # file piccoli → bassa latenza
            p.enable_deletion_vectors    = True
            p.log_retention_days         = 14
            p.vorder_enabled             = True
            p.enable_cdf                 = False
            p.notes = "Gold/Operational: file 64MB, DV, VORDER, compaction aggressiva."

        elif purpose == VisualizationPurpose.ML_TRAINING:
            # Feature store Gold — no V-ORDER, Z-ORDER su customer_id + timestamp
            p.run_optimize               = True
            p.zorder_columns             = zo       # es. ["customer_id", "feature_timestamp"]
            p.vorder_optimize            = False
            p.vacuum_retain_hours        = 720      # 30 giorni
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.auto_compact_min_files     = 10
            p.target_file_size_mb        = 256
            p.stats_columns              = 16
            p.log_retention_days         = 30
            p.vorder_enabled             = False    # ML non beneficia di V-ORDER
            p.enable_cdf                 = False
            p.optimize_hot_partitions_only = True
            p.hot_partition_days         = 30
            p.notes = (
                "Gold/ML Feature Store: NO V-ORDER, Z-ORDER su customer_id + timestamp, "
                "OPTIMIZE ultime 30gg."
            )

        elif purpose == VisualizationPurpose.ARCHIVE:
            p.run_optimize               = False
            p.run_vacuum                 = True
            p.vacuum_retain_hours        = 2160     # 90 giorni
            p.target_file_size_mb        = 1024
            p.log_retention_days         = 90
            p.vorder_enabled             = False
            p.auto_compact_enabled       = False
            p.enable_cdf                 = False
            p.notes = "Gold/Archive: NO OPTIMIZE, file 1GB, retention massima 90gg."

        else:
            p.run_optimize               = True
            p.zorder_columns             = zo
            p.vorder_optimize            = True
            p.vacuum_retain_hours        = 720
            p.auto_optimize              = True
            p.auto_compact_enabled       = True
            p.log_retention_days         = 30
            p.vorder_enabled             = True
            p.notes = "Gold: profilo generico serving layer."

    p.partition_hint = partition_column
    return p


# ─────────────────────────────────────────────
#  APPLICAZIONE DEL PROFILO
# ─────────────────────────────────────────────
def apply_optimization(
    spark: SparkSession,
    table_path: str,
    profile: OptimizationProfile,
    dry_run: bool = False,
) -> dict:
    """
    Applica il profilo di ottimizzazione a una tabella Delta.

    Parameters
    ----------
    spark      : SparkSession attiva
    table_path : nome tabella Hive o path ABFSS (es. "gold.sales" oppure "abfss://...")
    profile    : OptimizationProfile calcolato
    dry_run    : se True stampa le operazioni senza eseguirle
    """

    ops   = []
    warns = []

    log.info(f"{'[DRY RUN] ' if dry_run else ''}Ottimizzazione: {table_path}")
    log.info(f"  Layer note: {profile.notes}")

    # ── Warehouse mode: skip tutto ─────────────────────────────────────────
    if profile.warehouse_mode:
        log.info("    Warehouse mode: OPTIMIZE/VACUUM delegati a Fabric.")
        ops.append("SKIP (warehouse_mode=True)")
        return {"table": table_path, "operations": ops, "warnings": warns, "dry_run": dry_run}

    # ── Liquid Clustering warning ──────────────────────────────────────────
    if profile.enable_liquid_clustering and profile.table_size_gb and profile.table_size_gb < 100:
        msg = (
            f"Liquid Clustering su tabella {profile.table_size_gb:.1f}GB (<100GB): "
            "valuta Z-ORDER per costo minore."
        )
        log.warning(f"    {msg}")
        warns.append(msg)

    # ── Configura TBLPROPERTIES ────────────────────────────────────────────
    props = {}

    if profile.target_file_size_mb:
        props["delta.targetFileSize"] = str(profile.target_file_size_mb * 1024 * 1024)
    if profile.auto_optimize:
        props["delta.autoOptimize.optimizeWrite"] = "true"
    if profile.auto_compact:
        props["delta.autoOptimize.autoCompact"] = "true"
    if profile.enable_deletion_vectors:
        props["delta.enableDeletionVectors"] = "true"
    if profile.stats_columns:
        props["delta.dataSkippingNumIndexedCols"] = str(profile.stats_columns)
    if profile.auto_compact_enabled:
        props["delta.autoCompact.enabled"]     = "true"
        props["delta.autoCompact.minNumFiles"] = str(profile.auto_compact_min_files)
    if profile.vorder_enabled:
        props["delta.parquet.vorder.enabled"] = "true"
    if profile.log_retention_days:
        props["delta.logRetentionDuration"] = f"interval {profile.log_retention_days} days"
    if profile.enable_cdf:
        props["delta.enableChangeDataFeed"] = "true"
    if profile.enable_column_mapping:
        props["delta.columnMapping.mode"] = "name"
    if profile.checkpoint_interval:
        props["delta.checkpointInterval"] = str(profile.checkpoint_interval)

    if props and not dry_run:
        alter_sql = ", ".join([f"'{k}' = '{v}'" for k, v in props.items()])
        spark.sql(f"ALTER TABLE {table_path} SET TBLPROPERTIES ({alter_sql})")
        ops.append(f"SET TBLPROPERTIES: {list(props.keys())}")
        log.info(f"  SET TBLPROPERTIES: {props}")
    elif props:
        log.info(f"  [DRY RUN] SET TBLPROPERTIES: {props}")
        ops.append(f"[DRY RUN] SET TBLPROPERTIES: {list(props.keys())}")

    # ── Liquid Clustering ──────────────────────────────────────────────────
    if profile.enable_liquid_clustering and profile.liquid_cluster_columns:
        cluster_cols = ", ".join(profile.liquid_cluster_columns)
        sql = f"ALTER TABLE {table_path} CLUSTER BY ({cluster_cols})"
        log.info(f"  Liquid Clustering: {cluster_cols}")
        if not dry_run:
            spark.sql(sql)
        ops.append(f"CLUSTER BY ({cluster_cols})")

    # ── OPTIMIZE (con logica DRY RUN + hot partitions) ─────────────────────
    if profile.run_optimize:

        # DRY RUN check: salta OPTIMIZE se auto-compact ha già gestito la frammentazione
        should_optimize = True
        if profile.optimize_dry_run_first and not dry_run:
            dry = spark.sql(f"OPTIMIZE {table_path} DRY RUN")
            files_to_compact = dry.count()
            log.info(f"  DRY RUN: {files_to_compact} file da compattare")
            if files_to_compact <= 10:
                log.info("    Skipping OPTIMIZE: auto-compaction già sufficiente.")
                ops.append(f"OPTIMIZE SKIPPED (DRY RUN: solo {files_to_compact} file da compattare)")
                should_optimize = False

        if should_optimize:
            # Costruisci WHERE per hot partitions
            where_clause = ""
            if profile.optimize_hot_partitions_only and profile.partition_column:
                cutoff = (datetime.now() - timedelta(days=profile.hot_partition_days)).strftime('%Y-%m-%d')
                where_clause = f" WHERE {profile.partition_column} >= '{cutoff}'"
                log.info(f"  Hot partition filter: {profile.partition_column} >= {cutoff}")

            # Costruisci ORDER BY
            if profile.vorder_optimize:
                order_clause = " VORDER"
            elif profile.zorder_columns:
                order_clause = f" ZORDER BY ({', '.join(profile.zorder_columns)})"
            else:
                order_clause = ""

            sql = f"OPTIMIZE {table_path}{where_clause}{order_clause}"
            log.info(f"  {sql}")
            if not dry_run:
                spark.sql(sql)
            ops.append(sql)

    # ── VACUUM ─────────────────────────────────────────────────────────────
    if profile.run_vacuum:
        sql = f"VACUUM {table_path} RETAIN {profile.vacuum_retain_hours} HOURS"
        log.info(f"  {sql}")
        if not dry_run:
            spark.sql(sql)
        ops.append(sql)

    log.info("  ✓ Completato.\n")
    return {"table": table_path, "operations": ops, "warnings": warns, "dry_run": dry_run}


# ─────────────────────────────────────────────
#  VERIFICA PROPRIETÀ APPLICATE
# ─────────────────────────────────────────────

PROP_MAP = {
    "delta.targetFileSize":                lambda p: str(p.target_file_size_mb * 1024 * 1024) if p.target_file_size_mb else None,
    "delta.autoOptimize.optimizeWrite":    lambda p: "true" if p.auto_optimize else None,
    "delta.autoOptimize.autoCompact":      lambda p: "true" if p.auto_compact else None,
    "delta.enableDeletionVectors":         lambda p: "true" if p.enable_deletion_vectors else None,
    "delta.dataSkippingNumIndexedCols":    lambda p: str(p.stats_columns) if p.stats_columns else None,
    "delta.autoCompact.enabled":           lambda p: "true" if p.auto_compact_enabled else None,
    "delta.autoCompact.minNumFiles":       lambda p: str(p.auto_compact_min_files) if p.auto_compact_enabled else None,
    "delta.parquet.vorder.enabled":        lambda p: "true" if p.vorder_enabled else None,
    "delta.logRetentionDuration":          lambda p: f"interval {p.log_retention_days} days" if p.log_retention_days else None,
    "delta.enableChangeDataFeed":          lambda p: "true" if p.enable_cdf else None,
    "delta.columnMapping.mode":            lambda p: "name" if p.enable_column_mapping else None,
    "delta.checkpointInterval":            lambda p: str(p.checkpoint_interval) if p.checkpoint_interval else None,
}


@dataclass
class PropertyCheckResult:
    prop:     str
    expected: str
    actual:   Optional[str]

    @property
    def ok(self) -> bool:
        return self.actual == self.expected

    def __str__(self) -> str:
        status = "OK" if self.ok else "KO"
        actual_display = self.actual if self.actual is not None else "<non impostata>"
        return f"  {status} {self.prop}: atteso={self.expected}  effettivo={actual_display}"


def verify_table_properties(
    spark: SparkSession,
    table_path: str,
    profile: OptimizationProfile,
) -> list[PropertyCheckResult]:
    try:
        rows = spark.sql(f"SHOW TBLPROPERTIES {table_path}").collect()
        actual_props = {row["key"]: row["value"] for row in rows}
    except Exception as e:
        log.error(f"  Impossibile leggere TBLPROPERTIES per {table_path}: {e}")
        return []

    results = []
    for prop_key, expected_fn in PROP_MAP.items():
        expected_val = expected_fn(profile)
        if expected_val is None:
            continue
        actual_val = actual_props.get(prop_key)
        result = PropertyCheckResult(prop=prop_key, expected=expected_val, actual=actual_val)
        results.append(result)
        log.info(str(result)) if result.ok else log.warning(str(result))

    # Verifica Liquid Clustering
    if profile.enable_liquid_clustering and profile.liquid_cluster_columns:
        try:
            detail = spark.sql(f"DESCRIBE DETAIL {table_path}").collect()[0]
            actual_cc = detail["clusteringColumns"] or []
            expected_cc = profile.liquid_cluster_columns
            ok = sorted(actual_cc) == sorted(expected_cc)
            log.info(f"  {'OK' if ok else 'KO'} clusteringColumns: atteso={expected_cc}  effettivo={actual_cc}")
            results.append(PropertyCheckResult(
                prop="clusteringColumns",
                expected=str(sorted(expected_cc)),
                actual=str(sorted(actual_cc)),
            ))
        except Exception as e:
            log.warning(f"    Impossibile verificare Liquid Clustering: {e}")

    return results


def verify_and_report(spark, table_path, profile) -> dict:
    log.info(f"\n Verifica proprietà: {table_path}")
    checks = verify_table_properties(spark, table_path, profile)
    passed = sum(1 for c in checks if c.ok)
    failed = sum(1 for c in checks if not c.ok)
    summary = {"table": table_path, "checks": checks, "passed": passed, "failed": failed, "success": failed == 0}
    if failed == 0:
        log.info(f"  OK Tutti i {passed} check superati.")
    else:
        log.warning(f"    {failed} check falliti su {passed + failed} totali.")
    return summary


# ─────────────────────────────────────────────
#  MONITORING
# ─────────────────────────────────────────────

def table_health_score(spark: SparkSession, table_name: str) -> dict:
    """
    Calcola il health score di una tabella Delta basato su:
    - Coefficient of Variation (CV) della dimensione dei file
    - Numero file totali
    - Giorni dall'ultimo OPTIMIZE

    CV < 0.3 = GOOD  |  0.3-0.8 = WARNING  |  > 0.8 = CRITICAL
    """
    try:
        details = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
        num_files = details["numFiles"]
        size_bytes = details["sizeInBytes"]
        avg_file_mb = (size_bytes / num_files) / (1024 ** 2) if num_files > 0 else 0

        # Giorni dall'ultimo OPTIMIZE
        history = spark.sql(f"DESCRIBE HISTORY {table_name}").collect()
        last_optimize = next(
            (row["timestamp"] for row in history if row["operation"] == "OPTIMIZE"), None
        )
        days_since_optimize = (datetime.now() - last_optimize).days if last_optimize else 999

        # Stima CV dai file (richiede listing — usa num_files come proxy)
        # Valore proxy: se avg_file_mb < 50MB → alta frammentazione
        if avg_file_mb < 50:
            cv_proxy = 1.0
            health = "CRITICAL"
        elif avg_file_mb < 128:
            cv_proxy = 0.6
            health = "WARNING"
        else:
            cv_proxy = 0.2
            health = "GOOD"

        if days_since_optimize > 14:
            health = "WARNING" if health == "GOOD" else health

        return {
            "table": table_name,
            "num_files": num_files,
            "size_gb": size_bytes / (1024 ** 3),
            "avg_file_mb": avg_file_mb,
            "cv_proxy": cv_proxy,
            "days_since_optimize": days_since_optimize,
            "health": health,
        }
    except Exception as e:
        log.error(f"  Errore health_score su {table_name}: {e}")
        return {"table": table_name, "health": "ERROR", "error": str(e)}


def measure_write_amplification(spark: SparkSession, table_name: str) -> float:
    """
    Misura il write amplification ratio dell'ultimo OPTIMIZE.
    Ratio > 1.5 → segnala doppia compaction (auto-compact + OPTIMIZE schedulato).
    """
    try:
        history = spark.sql(f"DESCRIBE HISTORY {table_name}").limit(1).collect()[0]
        metrics = history["operationMetrics"]
        bytes_written = int(metrics.get("numOutputBytes", 0))
        bytes_read    = int(metrics.get("numInputBytes", 1))
        ratio = bytes_written / bytes_read
        if ratio > 1.5:
            log.warning(
                f"    Write amplification elevato su {table_name}: {ratio:.2f}x "
                f"(auto-compact potrebbe star rifacendo il lavoro di OPTIMIZE)"
            )
        return ratio
    except Exception as e:
        log.error(f"  Errore write_amplification su {table_name}: {e}")
        return 0.0


def deletion_vectors_health(spark: SparkSession, table_name: str) -> float:
    """
    Stima la percentuale di righe marcate come eliminate tramite Deletion Vectors.
    Se > 15%: consigliato OPTIMIZE per riscrivere i file e rimuovere le righe morte.
    """
    try:
        stats = spark.sql(f"""
            SELECT
                SUM(stats.numRecords) as total_records
            FROM (DESCRIBE DETAIL {table_name})
        """).collect()[0]

        history = spark.sql(f"DESCRIBE HISTORY {table_name}").limit(5).collect()
        deleted_rows = sum(
            int(row["operationMetrics"].get("numDeletedRows", 0))
            for row in history
            if row["operationMetrics"]
        )

        total = stats["total_records"] or 1
        deleted_pct = (deleted_rows / total) * 100

        if deleted_pct > 15:
            log.warning(
                f"    {table_name}: {deleted_pct:.1f}% righe stimate in DV — "
                f"esegui OPTIMIZE per ricompattare."
            )
        return deleted_pct
    except Exception as e:
        log.error(f"  Errore DV health su {table_name}: {e}")
        return 0.0


def lakehouse_health_dashboard(spark: SparkSession, tables: list[str]) -> None:
    """
    Stampa un dashboard di health per una lista di tabelle Delta.
    Evidenzia le tabelle che richiedono OPTIMIZE o VACUUM.
    """
    results = []
    for table in tables:
        score = table_health_score(spark, table)
        results.append(score)

    print("\n" + "=" * 80)
    print(f"{'LAKEHOUSE HEALTH DASHBOARD':^80}")
    print("=" * 80)
    print(f"{'Tabella':<35} {'Size GB':>8} {'Files':>7} {'Avg MB':>8} {'Days opt':>9} {'Health':<10}")
    print("-" * 80)

    for r in results:
        if r.get("health") == "ERROR":
            print(f"{r['table']:<35} {'ERROR'}")
            continue
        icon = "OK" if r["health"] == "GOOD" else (" " if r["health"] == "WARNING" else "KO")
        print(
            f"{r['table']:<35} "
            f"{r.get('size_gb', 0):>8.2f} "
            f"{r.get('num_files', 0):>7} "
            f"{r.get('avg_file_mb', 0):>8.1f} "
            f"{r.get('days_since_optimize', 999):>9} "
            f"{icon} {r['health']}"
        )

    critical = [r for r in results if r.get("health") in ("WARNING", "CRITICAL")]
    if critical:
        print(f"\n    {len(critical)} tabelle richiedono attenzione:")
        for r in critical:
            print(f"     → {r['table']} (avg_file={r.get('avg_file_mb',0):.0f}MB, days={r.get('days_since_optimize',999)})")
    print("=" * 80 + "\n")


# ─────────────────────────────────────────────
#  HELPER: ottimizza lista di tabelle
# ─────────────────────────────────────────────
def optimize_tables(
    spark: SparkSession,
    tables: list[dict],
    dry_run: bool = False,
    verify: bool = True,
) -> list[dict]:
    """
    Ottimizza una lista di tabelle con configurazione per ognuna.
    Se verify=True (default), esegue la verifica delle proprietà dopo ogni ottimizzazione.

    Ogni elemento di `tables` è un dict con:
        - path             (str)               : nome tabella o path ABFSS
        - layer            (MedallionLayer)
        - purpose          (VisualizationPurpose)
        - zorder_cols      (list[str], opz.)
        - partition_col    (str, opz.)         : colonna data per hot-partition filter
        - consumer_strategy (ConsumerStrategy, opz.)
        - warehouse_mode   (bool, opz.)
        - table_size_gb    (float, opz.)       : dimensione per decisioni Liquid Clustering
    """
    results = []
    for t in tables:
        profile = get_optimization_profile(
            layer=t["layer"],
            purpose=t["purpose"],
            zorder_columns=t.get("zorder_cols"),
            partition_column=t.get("partition_col"),
            consumer_strategy=t.get("consumer_strategy", ConsumerStrategy.SINGLE_TABLE),
            warehouse_mode=t.get("warehouse_mode", False),
            table_size_gb=t.get("table_size_gb"),
        )
        result = apply_optimization(spark, t["path"], profile, dry_run=dry_run)

        if verify and not dry_run:
            result["verification"] = verify_and_report(spark, t["path"], profile)
        else:
            result["verification"] = None

        results.append(result)
    return results


# ─────────────────────────────────────────────
#  ESEMPIO D'USO
# ─────────────────────────────────────────────
if __name__ == "__main__":

    spark = SparkSession.builder.appName("DeltaOptimizer").getOrCreate()
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

    tables_to_optimize = [

        # ── BRONZE ──────────────────────────────────────────────────────────
            # Bronze raw ingestion
        {
            "path": "abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries",
            "layer": MedallionLayer.BRONZE,
            "purpose": VisualizationPurpose.ARCHIVE,
        },
            # NO auto-compact, CDF abilitato, VACUUM 48h
        {
            "path":   "bronze.raw_eventi_streaming",
            "layer":  MedallionLayer.BRONZE,
            "purpose": VisualizationPurpose.OPERATIONAL,
            # optimizeWrite ON, NO OPTIMIZE schedulato
        },

        # # ── SILVER ──────────────────────────────────────────────────────────
        # {
        #     "path":          "silver.fact_vendite",
        #     "layer":         MedallionLayer.SILVER,
        #     "purpose":       VisualizationPurpose.ANALYTICAL,
        #     "zorder_cols":   ["DataOrdine", "CodiceCliente", "CodiceArticolo"],
        #     "partition_col": "DataOrdine",   # OPTIMIZE solo hot partitions 14gg
        #     # DRY RUN prima, file 256MB, NO V-ORDER
        # },
        # {
        #     "path":          "silver.eventi_produzione",
        #     "layer":         MedallionLayer.SILVER,
        #     "purpose":       VisualizationPurpose.OPERATIONAL,
        #     "zorder_cols":   ["EventTimestamp", "MacchinarioID"],
        #     "partition_col": "EventDate",
        # },
        # {
        #     "path":   "silver.features_clienti",
        #     "layer":  MedallionLayer.SILVER,
        #     "purpose": VisualizationPurpose.ML_TRAINING,
        #     # file 512MB, NO V-ORDER, NO Z-ORDER
        # },

        # # ── GOLD ────────────────────────────────────────────────────────────
        # {
        #     "path":              "gold.kpi_vendite_mensili",
        #     "layer":             MedallionLayer.GOLD,
        #     "purpose":           VisualizationPurpose.DASHBOARD_KPI,
        #     "zorder_cols":       ["Anno", "Mese", "BusinessUnit"],
        #     "partition_col":     "DataVendita",
        #     "consumer_strategy": ConsumerStrategy.SINGLE_TABLE,  # 400MB + VORDER
        #     "table_size_gb":     45.0,  # <100GB → Z-ORDER, non Liquid Clustering
        # },
        # {
        #     "path":              "gold.sales_fact_powerbi",
        #     "layer":             MedallionLayer.GOLD,
        #     "purpose":           VisualizationPurpose.DASHBOARD_KPI,
        #     "zorder_cols":       ["SaleDate", "Region"],
        #     "consumer_strategy": ConsumerStrategy.DEDICATED_TABLES,  # 600MB + VORDER
        #     "table_size_gb":     120.0,  # >100GB → Liquid Clustering
        # },
        # {
        #     "path":          "gold.ml_customer_features",
        #     "layer":         MedallionLayer.GOLD,
        #     "purpose":       VisualizationPurpose.ML_TRAINING,
        #     "zorder_cols":   ["customer_id", "feature_timestamp"],
        #     "partition_col": "feature_date",
        # },
        # {
        #     "path":   "gold.export_fatturato",
        #     "layer":  MedallionLayer.GOLD,
        #     "purpose": VisualizationPurpose.EXPORT,
        # },

        # # ── WAREHOUSE MODE ───────────────────────────────────────────────────
        # {
        #     "path":          "gold.warehouse_sales",
        #     "layer":         MedallionLayer.GOLD,
        #     "purpose":       VisualizationPurpose.DASHBOARD_KPI,
        #     "warehouse_mode": True,   # Fabric Warehouse: skip OPTIMIZE/VACUUM
        # },
    ]

    # ── Esegui ottimizzazioni ──────────────────────────────────────────────
    results = optimize_tables(spark, tables_to_optimize, dry_run=True, verify=False)

    print("\n" + "=" * 70)
    print("RIEPILOGO OTTIMIZZAZIONI")
    print("=" * 70)
    total_passed = 0
    total_failed = 0
    for r in results:
        print(f"\n {r['table']}")
        for w in r.get("warnings", []):
            print(f"     {w}")
        for op in r["operations"]:
            print(f"   → {op}")
        v = r.get("verification")
        if v:
            status_icon = "OK" if v["success"] else "KO"
            print(f"   {status_icon} Verifica: {v['passed']} OK, {v['failed']} KO")
            for chk in v["checks"]:
                if not chk.ok:
                    print(f"        {chk.prop}: atteso={chk.expected}, effettivo={chk.actual}")
            total_passed += v["passed"]
            total_failed += v["failed"]

    if total_passed + total_failed > 0:
        print("\n" + "=" * 70)
        print(f"TOTALE CHECK: {total_passed + total_failed}  OK {total_passed} OK  KO {total_failed} KO")
        print("=" * 70)

    # ── Health dashboard (esempio) ─────────────────────────────────────────
    # lakehouse_health_dashboard(spark, [
    #     "gold.kpi_vendite_mensili",
    #     "gold.sales_fact_powerbi",
    #     "silver.fact_vendite",
    # ])

StatementMeta(, d0c1a2aa-900f-475a-a1c4-3169eac98b30, 5, Finished, Available, Finished, True)


RIEPILOGO OTTIMIZZAZIONI

 abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries
   → [DRY RUN] SET TBLPROPERTIES: ['delta.targetFileSize', 'delta.dataSkippingNumIndexedCols', 'delta.logRetentionDuration', 'delta.enableChangeDataFeed', 'delta.columnMapping.mode']
   → VACUUM abfss://2cd8c8d6-210e-453e-a27d-453b9ffc21f1@onelake.dfs.fabric.microsoft.com/c3fcb4c1-59fb-408a-9972-b35608d80496/Tables/DB_Countries RETAIN 48 HOURS

 bronze.raw_eventi_streaming
   → [DRY RUN] SET TBLPROPERTIES: ['delta.targetFileSize', 'delta.autoOptimize.optimizeWrite', 'delta.dataSkippingNumIndexedCols', 'delta.logRetentionDuration', 'delta.enableChangeDataFeed', 'delta.columnMapping.mode']
   → VACUUM bronze.raw_eventi_streaming RETAIN 48 HOURS
